<a href="https://colab.research.google.com/github/YOUR-USERNAME/bags-vectors-transformers/blob/main/day1/notebooks/1_intro_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bags, Vectors & Transformers
## Day 1 — Working with Text Data in Python

**A Methods Workshop in Computational Text Analysis**
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

Welcome to the first hands-on notebook! By the end of it you will be able to:

- Load and explore text data in Python
- Clean and preprocess text (tokenization, lowercasing, stopwords, stemming/lemmatization)
- Count word frequencies
- Visualize text with bar charts and word clouds

You do **not** need to be a Python expert. We assume you have seen the basics before
(variables, lists, loops), roughly at the level of an introductory DataCamp course. If
something is unfamiliar, ask — that is what today is for.

> **How to use this notebook:** run each cell in order with `Shift + Enter`. Read the text,
> run the code, look at the output. Wherever you see a **✏️ Exercise**, have a go before moving on.


## 0. Setup

We start by installing and importing the libraries we need. In Colab most of these are
already available; the `nltk` data downloads are the main thing to run once per session.


In [ ]:
# Core libraries
import re
import string
from collections import Counter

import matplotlib.pyplot as plt

# NLTK for text preprocessing
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

# Download the NLTK data we need (safe to run more than once)
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")

print("Setup complete!")

## 1. Text is just data

At its core, text in Python is stored as **strings**. Let's start with a single short
document — imagine it is one response to an open survey question.


In [ ]:
response = "I really love this course! The examples are great and super helpful :)"

print(response)
print()
print("Type:", type(response))
print("Length (characters):", len(response))

A string is a sequence of characters. We can already do useful things with it using
plain Python — no special libraries required.


In [ ]:
# Make everything lowercase
print(response.lower())

# Split into words on whitespace
print(response.split())

# Count how many words (roughly)
print("Rough word count:", len(response.split()))

> **✏️ Exercise 1**
>
> Create your own string variable called `my_text` containing a sentence about your
> research. Then print it in **uppercase** and print how many characters it has.
> *(Hint: strings have an `.upper()` method, and `len()` gives length.)*


In [ ]:
# Your code here


## 2. From one document to a corpus

A single text is rarely interesting on its own. In computational text analysis we work
with a **corpus**: a collection of documents. In Python, the simplest way to hold a
corpus is a **list of strings**.

Below is our toy corpus for today: a handful of made-up tweets about a (fictional) new
public transport policy. Small enough to read by eye, which makes it perfect for learning.


In [ ]:
corpus = [
    "I love the new bus policy, it makes my commute so much easier!",
    "The new bus policy is a disaster. Waited 40 minutes today. Awful.",
    "Not sure how I feel about the new transport plan yet. We will see.",
    "Great to see the city investing in public transport. Long overdue!",
    "The bus policy is terrible and expensive. Who approved this??",
    "Cycling to work now instead of the bus. The new plan is useless.",
    "Honestly the new buses are clean and fast. I am impressed!",
    "More buses, fewer cars. This is exactly what we needed.",
]

print(f"Our corpus has {len(corpus)} documents.\n")
for i, doc in enumerate(corpus):
    print(f"D{i+1}: {doc}")

> **✏️ Exercise 2**
>
> Loop over the corpus and print the **length in characters** of each document.
> Which document is the longest?


In [ ]:
# Your code here


## 3. Tokenization

Before we can count or analyze words, we need to split text into **tokens** — the
individual units (usually words). This is called **tokenization**.

We saw that `.split()` gives a rough version. But it has problems: punctuation sticks to
words (`"easier!"`), and it does not handle edge cases well. Proper tokenizers, like the
one in NLTK, are smarter.


In [ ]:
example = corpus[0]
print("Original:", example)
print()

# Naive: split on whitespace
print("Naive split:  ", example.split())
print()

# NLTK tokenizer
print("NLTK tokens:  ", word_tokenize(example))

Notice how the NLTK tokenizer separates punctuation into its own tokens (e.g. `easier`
and `!` become separate). That is usually what we want, because it stops `"easier!"` and
`"easier"` from being treated as two different words.

**Recap of a key concept from the lecture:**
- A **token** is each occurrence of a word.
- A **type** is each *distinct* word.


In [ ]:
tokens = word_tokenize(corpus[0].lower())
print("Tokens:", tokens)
print()
print("Number of tokens (total words):", len(tokens))
print("Number of types (distinct words):", len(set(tokens)))

> **✏️ Exercise 3**
>
> Tokenize the **second** document in the corpus (remember Python indexing starts at 0).
> How many tokens does it have? How many types?


In [ ]:
# Your code here


## 4. Preprocessing: cleaning up the text

Raw tokens are messy. Before counting, we usually **normalize** the text. Each step is a
**decision** that changes what our numbers mean — so think about whether each one suits
your research question.

We will build up a cleaning pipeline step by step.


### 4.1 Lowercasing

`"The"` and `"the"` are the same word for most purposes, so we lowercase everything.
(But note: sometimes case matters — e.g. `"US"` the country vs `"us"` the pronoun.)


In [ ]:
sample = "The BUS is Great but the Policy is Terrible"
print(sample.lower())

### 4.2 Removing punctuation

Punctuation tokens (`!`, `?`, `.`) usually are not what we want to count. Let's remove them.


In [ ]:
tokens = word_tokenize("The bus policy is terrible and expensive. Who approved this??".lower())
print("Before:", tokens)

# Keep only tokens that are alphabetic (drops punctuation and numbers)
tokens_no_punct = [t for t in tokens if t.isalpha()]
print("After: ", tokens_no_punct)

### 4.3 Removing stopwords

**Stopwords** are extremely common words (`the`, `is`, `and`, `of`...) that usually carry
little topical meaning. Removing them focuses attention on content words.

⚠️ **Be careful:** stopwords can matter! Removing `not` turns *"not good"* into *"good"*.
Always consider your question before stripping them.


In [ ]:
stop_words = set(stopwords.words("english"))
print("A few English stopwords:", list(stop_words)[:15])
print("Total stopwords:", len(stop_words))

In [ ]:
tokens = word_tokenize("the bus policy is terrible and expensive".lower())
tokens_no_stop = [t for t in tokens if t not in stop_words]

print("Before:", tokens)
print("After: ", tokens_no_stop)

### 4.4 Stemming vs. lemmatization

Different forms of a word (`run`, `runs`, `running`) often mean the same thing. Two ways
to collapse them:

- **Stemming**: chops off endings using crude rules. Fast but can produce non-words
  (`"policy"` → `"polici"`).
- **Lemmatization**: maps a word to its dictionary form (its *lemma*). Slower but cleaner
  (`"better"` → `"good"` with the right settings).


In [ ]:
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

words = ["running", "runs", "ran", "easily", "buses", "policies", "better"]

print(f"{'word':<12}{'stem':<12}{'lemma':<12}")
print("-" * 36)
for w in words:
    print(f"{w:<12}{stemmer.stem(w):<12}{lemmatizer.lemmatize(w):<12}")

Notice the trade-offs: stemming turns `buses` into `buse`, while lemmatization correctly
gives `bus`. But lemmatization needs to know the part of speech to handle cases like
`better` → `good`.

> **✏️ Exercise 4**
>
> Try stemming and lemmatizing these words: `["studies", "studying", "cars", "communication"]`.
> Which method do you find gives more sensible results here?


In [ ]:
# Your code here


## 5. Putting it together: a preprocessing function

Let's combine everything into a single reusable function. This is the kind of function
you will write again and again in your own projects.


In [ ]:
def preprocess(text, remove_stopwords=True, do_lemmatize=True):
    """Clean a single document and return a list of tokens."""
    # 1. Lowercase
    text = text.lower()
    # 2. Tokenize
    tokens = word_tokenize(text)
    # 3. Keep only alphabetic tokens (drops punctuation and numbers)
    tokens = [t for t in tokens if t.isalpha()]
    # 4. Remove stopwords (optional)
    if remove_stopwords:
        tokens = [t for t in tokens if t not in stop_words]
    # 5. Lemmatize (optional)
    if do_lemmatize:
        tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return tokens


# Test it on the first document
print("Original: ", corpus[0])
print("Cleaned:  ", preprocess(corpus[0]))

Now we can clean the whole corpus in one line using a **list comprehension**.


In [ ]:
cleaned_corpus = [preprocess(doc) for doc in corpus]

for i, tokens in enumerate(cleaned_corpus):
    print(f"D{i+1}: {tokens}")

> **✏️ Exercise 5**
>
> Run `preprocess` on the disaster tweet (document index 1) **twice**: once with
> `remove_stopwords=True` and once with `remove_stopwords=False`. What is the difference?
> Can you find a case in our corpus where removing stopwords loses important meaning?


In [ ]:
# Your code here


## 6. Counting word frequencies

Now that our text is clean, we can count words. The `Counter` class from Python's
`collections` module makes this easy.

First, let's flatten our corpus (a list of lists) into a single list of all tokens.


In [ ]:
# Flatten: combine all documents' tokens into one big list
all_tokens = []
for tokens in cleaned_corpus:
    all_tokens.extend(tokens)

print("Total tokens across the corpus:", len(all_tokens))
print("First 20:", all_tokens[:20])

In [ ]:
# Count word frequencies
word_counts = Counter(all_tokens)

# The 10 most common words
print("Most common words:")
for word, count in word_counts.most_common(10):
    print(f"  {word:<12} {count}")

> **✏️ Exercise 6**
>
> Use `word_counts` to answer: how many times does the word `"bus"` appear? And `"policy"`?
> *(Hint: you can index a Counter like a dictionary, e.g. `word_counts["bus"]`.)*


In [ ]:
# Your code here


## 7. Visualizing word frequencies

A table of counts is fine, but a **bar chart** is easier to read. We'll use `matplotlib`.


In [ ]:
# Get the top 10 words and their counts
top_words = word_counts.most_common(10)
words = [w for w, c in top_words]
counts = [c for w, c in top_words]

plt.figure(figsize=(10, 5))
plt.bar(words, counts, color="#34B233")     # WUR green :)
plt.title("Top 10 words in the corpus")
plt.xlabel("Word")
plt.ylabel("Frequency")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

> **✏️ Exercise 7**
>
> Change the chart to show the top **15** words instead of 10. Then try changing the
> color. *(Matplotlib accepts color names like `"steelblue"` or hex codes like `"#1A1A2E"`.)*


In [ ]:
# Your code here


## 8. Word clouds

A **word cloud** shows words sized by frequency. They are eye-catching and good for
presentations — though be a little careful, as they can be more decorative than rigorous.
Use them to get a quick *feel* for a corpus, not as your final analysis.


In [ ]:
# The wordcloud library is usually pre-installed in Colab.
# If not, uncomment the next line:
# !pip install wordcloud

from wordcloud import WordCloud

# WordCloud expects a single string of space-separated words
text_for_cloud = " ".join(all_tokens)

wc = WordCloud(
    width=800,
    height=400,
    background_color="white",
    colormap="Greens",
).generate(text_for_cloud)

plt.figure(figsize=(12, 6))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.title("Word cloud of the corpus")
plt.show()

> **✏️ Exercise 8**
>
> Generate a word cloud using only the tokens from the **positive** tweets
> (documents 0, 3, 6, 7). Do the prominent words look different from the full corpus?
>
> *(Hint: build a list of tokens from just those documents, then join them into a string.)*


In [ ]:
# Your code here


## Wrap-up

Nicely done! In this notebook you have:

- Loaded a corpus as a list of strings
- Tokenized text and distinguished **tokens** from **types**
- Built a preprocessing pipeline: lowercasing, punctuation removal, stopwords, lemmatization
- Counted word frequencies with `Counter`
- Visualized results with a **bar chart** and a **word cloud**

These are the foundations. In the next notebook we turn word counts into a
**document-term matrix** and start doing real analysis with it.

### Optional challenge

Write a function `top_words_for(doc_index, n=5)` that takes a document's index and returns
its `n` most common words *after preprocessing*. Test it on a few documents. This combines
almost everything from today.


In [ ]:
# Optional challenge — your code here
